In [6]:
import pickle
with open('best_model.pkl','rb') as file:
    model = pickle.load(file)
print("done")

done


In [5]:
import joblib
model = joblib.load('best_model.pkl')

<h1> Load Saved Model & Predict on Random Samples</h1>

This is a **separate notebook** that loads the model saved from
`01_netflix_ml_project.ipynb` and applies it to random rows from the processed dataset.

# Important Libraries

In [1]:
import pandas as pd
import numpy as np
import pickle
import os


> Make sure `label_type.pkl`, `feature_cols.pkl`, `scaler.pkl`, `netflix_processed.csv`, and either `best_model.pkl` (sklearn) OR `best_model_ann.keras` (Keras) are in the same folder.

# Load the saved model and preprocessing objects
We check which file exists to know whether the best model was a scikit-learn model or the Keras ANN.

In [ ]:
with open('label_type.pkl', 'rb') as f:
    label_type = pickle.load(f)

with open('feature_cols.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

if os.path.exists('best_model_ann.keras'):
    from tensorflow.keras.models import load_model
    model = load_model('best_model_ann.keras')
    is_keras = True
    print("Loaded Keras ANN model")
else:
    with open('best_model.pkl', 'rb') as f:
        model = pickle.load(f)
    is_keras = False
    print("Model loaded:", type(model).__name__)


# Load the processed data and take random samples

In [ ]:
df = pd.read_csv('netflix_processed.csv')
df.head()


In [ ]:
sample = df.sample(10).reset_index(drop=True)   # random samples every run
sample[['title'] + feature_cols]


### Predict

In [ ]:
x_sample = sample[feature_cols]

if is_keras:
    # Keras model was trained on scaled features and outputs one probability per class
    x_sample_scaled = scaler.transform(x_sample)
    y_pred = np.argmax(model.predict(x_sample_scaled), axis=1)
else:
    # Tree-based models (Decision Tree / Random Forest / AdaBoost / XGBoost) don't need scaling.
    # Logistic Regression / SVM / KNN / MLP need the SAME scaler used in training.
    needs_scaling = type(model).__name__ in ['LogisticRegression', 'SVC', 'KNeighborsClassifier', 'MLPClassifier']
    x_sample_for_pred = scaler.transform(x_sample) if needs_scaling else x_sample
    y_pred = model.predict(x_sample_for_pred)

predicted_type = label_type.inverse_transform(y_pred)

sample_result = sample[['title', 'type']].copy()
sample_result['type'] = label_type.inverse_transform(sample_result['type'])
sample_result['predicted_type'] = predicted_type
sample_result['correct'] = sample_result['type'] == sample_result['predicted_type']
sample_result


In [ ]:
accuracy_on_sample = sample_result['correct'].mean()
print(f"Accuracy on this random sample: {accuracy_on_sample:.2%}")
